# 20 - Sesion especial: simulacion en programacion, de variable aleatoria a SIR con POO

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que significa simular un sistema en programacion.
2. Conectar el concepto de variable aleatoria con codigo ejecutable.
3. Construir distribuciones empiricas a partir de muchas repeticiones.
4. Distinguir entre resultado de una corrida y comportamiento promedio.
5. Modelar un proceso dinamico paso a paso.
6. Implementar un modelo SIR estocastico.
7. Encapsular la simulacion del SIR con Programacion Orientada a Objetos.
8. Detectar errores comunes de modelado, interpretacion y programacion.


## Ruta de la sesion (secuencia ideal)

1. Que es simular.
2. Azar computacional y semillas.
3. Variable aleatoria (Bernoulli y conteos).
4. Estimacion por Monte Carlo.
5. Del experimento aislado al sistema con estado.
6. Modelo SIR estocastico (primero procedural).
7. Refactor a POO con una clase `SIRSimulador`.
8. Analisis de escenarios.
9. Errores comunes.
10. Ejercicios de pensamiento y diseno.


## 1. Que significa simular en programacion

Simular no es "adivinar" ni "dibujar datos".
Simular es:

- definir reglas explicitas de un sistema,
- ejecutar esas reglas muchas veces,
- observar resultados posibles bajo incertidumbre.

Cuando el sistema tiene azar, una sola corrida no representa toda la historia.
Por eso se repite, se resume y se interpreta.


## 2. Azar computacional: pseudoaleatorio y semilla

En computadora usamos generadores pseudoaleatorios.
Con la misma semilla (`seed`) obtenemos la misma secuencia, util para depurar y comparar.


In [ ]:
import random
from statistics import mean

# Creamos dos generadores pseudoaleatorios con la misma semilla.
# Esto garantiza que produzcan exactamente la misma secuencia.
rng_a = random.Random(2026)
rng_b = random.Random(2026)

# Generamos 5 numeros en [0, 1) con cada generador.
# round(..., 4) solo mejora la lectura en pantalla.
print([round(rng_a.random(), 4) for _ in range(5)])
print([round(rng_b.random(), 4) for _ in range(5)])


## 3. Variable aleatoria: primer modelo con Bernoulli

Una variable aleatoria asigna un numero a cada resultado posible de un experimento.
Ejemplo Bernoulli:

- `1` si ocurre evento (contagio, compra, exito),
- `0` si no ocurre.

Eso ya permite medir probabilidades, promedios y riesgos.


In [ ]:
def bernoulli(p, rng):
    # Validamos que p sea una probabilidad valida.
    if not 0 <= p <= 1:
        raise ValueError("p debe estar entre 0 y 1")

    # Regla Bernoulli: regresa 1 con probabilidad p, 0 en otro caso.
    return 1 if rng.random() < p else 0

# Instancia local del generador para que el experimento sea reproducible.
rng = random.Random(123)

# Tomamos 20 muestras de una Bernoulli(0.3).
muestras = [bernoulli(0.3, rng) for _ in range(20)]

# Mostramos resultados y su media empirica (aproxima la probabilidad p).
print("Muestras:", muestras)
print("Promedio empirico:", round(mean(muestras), 3))


## 4. Distribucion empirica: repetir para entender comportamiento

Una corrida puede salir rara.
Muchas corridas revelan patrones estables.


In [ ]:
def conteo_exitos(n, p, rng):
    # Suma n variables Bernoulli para contar exitos totales.
    return sum(bernoulli(p, rng) for _ in range(n))

# Repetimos muchas veces el experimento para construir distribucion empirica.
rng = random.Random(7)
resultados = [conteo_exitos(10, 0.4, rng) for _ in range(5000)]

# Diccionario: valor observado -> frecuencia.
frecuencias = {}
for x in resultados:
    frecuencias[x] = frecuencias.get(x, 0) + 1

# Imprimimos la distribucion de X = numero de exitos en 10 ensayos.
for valor in sorted(frecuencias):
    print(f"X={valor}: {frecuencias[valor]}")

# Reporte de tendencia central del experimento.
print("Media de X:", round(mean(resultados), 3))


## 5. Monte Carlo: estimar probabilidades por simulacion

Problema: ?cual es la probabilidad de tener al menos 2 fallas en 8 componentes,
si cada componente falla con probabilidad 0.15 de forma independiente?

Puedes resolverlo con formula cerrada o con simulacion.
La simulacion es clave cuando la formula exacta no es sencilla.


In [ ]:
from math import comb


def estimar_prob_evento(evento, repeticiones, rng):
    # Monte Carlo: cuenta cuantas veces ocurre el evento en muchas corridas.
    exitos = 0
    for _ in range(repeticiones):
        if evento(rng):
            exitos += 1
    return exitos / repeticiones


def evento_fallas(rng):
    # Simulamos 8 componentes (0/1: no falla/falla) y contamos fallas.
    fallas = sum(bernoulli(0.15, rng) for _ in range(8))
    return fallas >= 2


def prob_exacto_al_menos_2_fallas():
    # Calculamos la probabilidad exacta usando complemento:
    # P(X >= 2) = 1 - P(X=0) - P(X=1), con X ~ Binomial(8, 0.15).
    p = 0.15
    n = 8
    p0 = comb(n, 0) * (p ** 0) * ((1 - p) ** n)
    p1 = comb(n, 1) * (p ** 1) * ((1 - p) ** (n - 1))
    return 1 - (p0 + p1)


# Comparamos estimacion por simulacion contra valor exacto.
rng = random.Random(99)
p_estimada = estimar_prob_evento(evento_fallas, 50000, rng)
p_exacta = prob_exacto_al_menos_2_fallas()

print("Probabilidad estimada:", round(p_estimada, 4))
print("Probabilidad exacta:", round(p_exacta, 4))
print("Error absoluto:", round(abs(p_estimada - p_exacta), 4))


## 6. De experimento aislado a sistema dinamico

Un sistema dinamico tiene estado que cambia en el tiempo.
En epidemiologia, ese estado puede ser cuantas personas estan:

- Susceptibles (`S`),
- Infectadas (`I`),
- Recuperadas (`R`).

Eso es el nucleo del modelo SIR.


## 7. Modelo SIR estocastico (version procedural minima)

Primero lo implementamos de forma procedural para entender reglas.
Luego lo pasamos a POO para mejorar claridad y mantenibilidad.


In [ ]:
def binomial_por_ensayos(n, p, rng):
    # Version simple de Binomial(n, p) por conteo de ensayos Bernoulli.
    if n < 0:
        raise ValueError("n no puede ser negativo")
    if not 0 <= p <= 1:
        raise ValueError("p debe estar entre 0 y 1")

    exitos = 0
    for _ in range(n):
        if rng.random() < p:
            exitos += 1
    return exitos


def simular_sir_procedural(poblacion, infectados_iniciales, beta, gamma, pasos, semilla=None):
    # Validaciones basicas de parametros.
    if poblacion <= 0:
        raise ValueError("poblacion debe ser positiva")
    if not 0 < infectados_iniciales <= poblacion:
        raise ValueError("infectados_iniciales debe estar en (0, poblacion]")
    if not 0 <= gamma <= 1:
        raise ValueError("gamma debe estar entre 0 y 1")
    if not 0 <= beta <= poblacion:
        raise ValueError("beta debe estar entre 0 y poblacion")

    # Estado inicial del sistema SIR.
    rng = random.Random(semilla)
    s = poblacion - infectados_iniciales
    i = infectados_iniciales
    r = 0

    # Guardamos el historial como tuplas: (tiempo, S, I, R).
    historial = [(0, s, i, r)]

    # Evolucion paso a paso.
    for t in range(1, pasos + 1):
        # Probabilidad de que un susceptible se contagie en este paso.
        p_contagio = 1 - (1 - beta / poblacion) ** i

        # Transiciones estocasticas del paso actual.
        nuevos_contagios = binomial_por_ensayos(s, p_contagio, rng)
        nuevas_recuperaciones = binomial_por_ensayos(i, gamma, rng)

        # Actualizamos el estado segun las transiciones.
        s -= nuevos_contagios
        i += nuevos_contagios - nuevas_recuperaciones
        r += nuevas_recuperaciones

        historial.append((t, s, i, r))

    return historial


# Corrida de ejemplo con parametros fijos.
hist = simular_sir_procedural(
    poblacion=200,
    infectados_iniciales=3,
    beta=0.9,
    gamma=0.2,
    pasos=30,
    semilla=11,
)

print("Primeros 5 estados:")
for fila in hist[:5]:
    print(fila)


## 8. Limitacion de la version procedural

La version procedural funciona, pero mezcla:

- validaciones,
- estado,
- reglas de transicion,
- registro de historial.

Con POO podemos encapsular todo eso en una sola abstraccion coherente.


## 9. Aplicacion final: modelo SIR con Programacion Orientada a Objetos


In [ ]:
from dataclasses import dataclass


# Estructura inmutable para guardar un estado puntual del sistema.
@dataclass(frozen=True)
class EstadoSIR:
    t: int
    s: int
    i: int
    r: int


class SIRSimulador:
    def __init__(self, poblacion, infectados_iniciales, beta, gamma, semilla=None):
        # Parametros de configuracion del modelo.
        self.poblacion = poblacion
        self.infectados_iniciales = infectados_iniciales
        self.beta = beta
        self.gamma = gamma

        # Generador local para reproducibilidad y aislamiento del azar.
        self._rng = random.Random(semilla)

        # Aseguramos que el objeto nazca en estado valido.
        self._validar_parametros()
        self.reiniciar()

    def _validar_parametros(self):
        # Validaciones del dominio SIR.
        if self.poblacion <= 0:
            raise ValueError("poblacion debe ser positiva")
        if not 0 < self.infectados_iniciales <= self.poblacion:
            raise ValueError("infectados_iniciales debe estar en (0, poblacion]")
        if not 0 <= self.gamma <= 1:
            raise ValueError("gamma debe estar entre 0 y 1")
        if not 0 <= self.beta <= self.poblacion:
            raise ValueError("beta debe estar entre 0 y poblacion")

    def reiniciar(self):
        # Estado inicial y reinicio del historial.
        self.t = 0
        self.s = self.poblacion - self.infectados_iniciales
        self.i = self.infectados_iniciales
        self.r = 0
        self.historial = [EstadoSIR(self.t, self.s, self.i, self.r)]

    def _binomial(self, n, p):
        # Muestreador Binomial(n, p) por ensayos Bernoulli.
        exitos = 0
        for _ in range(n):
            if self._rng.random() < p:
                exitos += 1
        return exitos

    def paso(self):
        # Probabilidad de contagio de cada susceptible en este paso.
        p_contagio = 1 - (1 - self.beta / self.poblacion) ** self.i

        # Personas que cambian de compartimento.
        nuevos_contagios = self._binomial(self.s, p_contagio)
        nuevas_recuperaciones = self._binomial(self.i, self.gamma)

        # Actualizacion de compartimentos.
        self.s -= nuevos_contagios
        self.i += nuevos_contagios - nuevas_recuperaciones
        self.r += nuevas_recuperaciones

        # Avanza el reloj y registra el nuevo estado.
        self.t += 1
        estado = EstadoSIR(self.t, self.s, self.i, self.r)
        self.historial.append(estado)
        return estado

    def simular(self, pasos):
        # Ejecuta varios pasos consecutivos.
        for _ in range(pasos):
            self.paso()
        return list(self.historial)

    def resumen(self):
        # Resume indicadores utiles para comparar escenarios.
        pico = max(self.historial, key=lambda e: e.i)
        return {
            "pasos": self.t,
            "pico_infectados": pico.i,
            "dia_pico": pico.t,
            "infectados_final": self.i,
            "recuperados_final": self.r,
        }


In [ ]:
# Configuramos un escenario base del modelo.
sim = SIRSimulador(
    poblacion=300,
    infectados_iniciales=4,
    beta=0.8,
    gamma=0.18,
    semilla=2026,
)

# Simulamos 80 pasos de tiempo.
historial = sim.simular(80)

# Mostramos estado final y resumen de metricas.
print("Ultimo estado:", historial[-1])
print("Resumen:", sim.resumen())


## 10. Comparar escenarios (pensar en decisiones, no solo ejecutar)

Cambiar `beta` y `gamma` cambia la dinamica.
La pregunta util no es "que salida me dio", sino:

- ?por que ese patron?
- ?que supuesto del modelo lo explica?
- ?que decision cambiaria el resultado?


In [ ]:
def correr_escenario(beta, gamma, semilla):
    # Cada escenario usa mismos tamano poblacional e iniciales,
    # pero cambia parametros epidemiologicos.
    sim = SIRSimulador(
        poblacion=300,
        infectados_iniciales=4,
        beta=beta,
        gamma=gamma,
        semilla=semilla,
    )
    sim.simular(80)
    return sim.resumen()


# Lista de escenarios para comparar sensibilidad.
escenarios = [
    (0.6, 0.18, 10),
    (0.9, 0.18, 10),
    (0.9, 0.30, 10),
]

# Ejecutamos todos y reportamos metricas clave.
for beta, gamma, semilla in escenarios:
    resumen = correr_escenario(beta, gamma, semilla)
    print(f"beta={beta}, gamma={gamma} -> {resumen}")


## 11. Visualizacion rapida (opcional con matplotlib)

Si `matplotlib` esta disponible, puedes graficar.
Si no, el codigo sigue funcionando y puedes inspeccionar tablas.


In [ ]:
try:
    import matplotlib.pyplot as plt

    # Extraemos cada serie desde el historial para graficar.
    t = [e.t for e in historial]
    s = [e.s for e in historial]
    i = [e.i for e in historial]
    r = [e.r for e in historial]

    # Dibujamos la evolucion temporal de S, I y R.
    plt.figure(figsize=(9, 4))
    plt.plot(t, s, label="S")
    plt.plot(t, i, label="I")
    plt.plot(t, r, label="R")
    plt.title("Simulacion SIR estocastica")
    plt.xlabel("Paso")
    plt.ylabel("Personas")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
except ImportError:
    # El notebook sigue siendo util aunque no haya libreria de graficas.
    print("matplotlib no esta instalada. Puedes seguir con analisis numerico.")


## 12. Errores comunes al simular (remarcados)

1. Confundir una corrida con una conclusion general.
2. No fijar semilla cuando quieres depurar o comparar cambios.
3. Fijar semilla *dentro* de cada iteracion (rompe la variabilidad).
4. No validar parametros (probabilidades fuera de `[0, 1]`, poblacion negativa).
5. Olvidar invariantes del sistema (`S + I + R` debe mantenerse constante).
6. Mezclar unidades de tiempo sin declararlo (dias vs semanas).
7. Ajustar parametros hasta que "se vea bien" sin justificar supuestos.
8. Usar simulacion cuando hay solucion exacta simple y no explicarla.
9. Ignorar incertidumbre: reportar solo promedio sin dispersion.
10. Crear clases que solo almacenan datos sin encapsular reglas de negocio.


## 13. Ejercicios de pensamiento y practica real

Estos ejercicios buscan razonamiento de modelado.
No basta con que el codigo "corra": debes justificar supuestos y decisiones.


### Ejercicio 1: variable aleatoria bien definida

Disena una variable aleatoria `X` para "retraso de autobus" en minutos.

1. Define claramente el experimento aleatorio.
2. Propone dos distribuciones distintas para `X` y justifica cual usar.
3. Explica como cambiaria `E[X]` si hay hora pico.


In [ ]:
# Escribe aqui tu definicion de X y tu justificacion.


### Ejercicio 2: tamano de muestra y confianza

Quieres estimar por simulacion la probabilidad de que `I >= 40` antes del dia 50.

1. ?Cuantas corridas haras y por que?
2. ?Que metrica de estabilidad usaras para decidir si ya es suficiente?
3. Escribe pseudocodigo antes de programar.


In [ ]:
# Escribe aqui tu estrategia y luego implementa.


### Ejercicio 3: detectar un bug de aleatoriedad

Analiza el siguiente codigo y explica por que esta mal.
Luego corrigelo.


In [ ]:
# Bug intencional: la semilla se reinicia en cada corrida,
# por eso todas las simulaciones salen iguales.
import random

promedios = []
for _ in range(5):
    random.seed(42)  # <- error de diseno para experimentos independientes
    datos = [1 if random.random() < 0.3 else 0 for _ in range(100)]
    promedios.append(sum(datos) / len(datos))

print(promedios)


In [ ]:
# Corrige aqui el bug y explica el efecto sobre los resultados.


### Ejercicio 4: invariantes del modelo SIR

Antes de tocar codigo, escribe al menos 4 invariantes de `SIRSimulador`.
Ejemplos esperados: no negatividad, conservacion de poblacion, etc.

Luego agrega `assert` o validaciones para hacerlas ejecutables.


In [ ]:
# Implementa aqui tus invariantes ejecutables.


### Ejercicio 5: extender el modelo con vacunacion

Agrega una regla: cada paso se vacunan `v` personas susceptibles y pasan a `R`.

1. Decide donde vive esa regla (dentro de la clase o como estrategia externa).
2. Implementa la extension sin romper el API actual.
3. Explica tradeoffs de tu diseno.


In [ ]:
class SIRConVacunacion(SIRSimulador):
    # Implementa aqui tu extension.
    pass


### Ejercicio 6: separar politicas de contagio y recuperacion

Refactoriza para desacoplar reglas de contagio/recuperacion del simulador.
Pista: usa objetos estrategia o funciones inyectadas.

Objetivo: probar facil distintos supuestos sin reescribir la clase principal.


In [ ]:
# Esboza aqui interfaces o funciones de politica.


### Ejercicio 7: interpretar, no solo graficar

Corre 30 simulaciones con mismos parametros y semillas distintas.

1. Reporta promedio y desviacion estandar del pico de infectados.
2. Explica que significa alta dispersion para una decision publica.
3. ?Que recomendarias a una autoridad si el promedio se ve "aceptable" pero la cola de riesgo es alta?


In [ ]:
# Implementa aqui el experimento y su interpretacion.


### Ejercicio 8: limites del modelo

Escribe una critica tecnica del SIR usado aqui.

Incluye:
1. dos supuestos poco realistas,
2. una variable relevante que falta,
3. una mejora concreta y como la modelarias.


In [ ]:
# Responde aqui en comentarios o markdown dentro de la celda.


## 14. Resumen de conceptos clave

1. Simular es ejecutar reglas explicitas bajo incertidumbre.
2. Una variable aleatoria es una forma formal de representar resultados numericos de un experimento.
3. Monte Carlo aproxima probabilidades cuando repetir es mas simple que derivar formula.
4. En sistemas dinamicos importa la evolucion temporal del estado, no solo un valor final.
5. El modelo SIR separa poblacion en `S`, `I` y `R` para estudiar contagio y recuperacion.
6. POO ayuda a encapsular estado, invariantes y reglas de transicion.
7. Las conclusiones deben considerar variabilidad entre corridas, no solo una salida.


## 15. Reto final opcional

Construye una mini investigacion reproducible con tu `SIRSimulador`:

1. Define una pregunta de politica publica (por ejemplo: "?cuanto reducir `beta` para bajar el pico bajo 20?").
2. Corre al menos 50 simulaciones por escenario.
3. Compara tres estrategias (sin intervencion, reduccion de contacto, aumento de recuperacion).
4. Presenta resultados con una conclusion argumentada y limites del analisis.
